In [ ]:
"""
This project is meant to simulates a degree audit system from the
Dominican House of Studies. This system consists of student and what program are they in (students).
The course work for each program (programs). The programs within the programs dataframe
consists of STB, MA Theology, Master of Divinity,MA Thomistic Studies, STL, and STD.
The required coursework that are needed to graduate with the degree (program_requirements).
The students and the courses they have taken or not taken so far (enrollments). The courses they are taking
each have a certain category depending on the course work.

Pipeline:
1. Merge student, course, and enrollment data
2. Filter for valid (completed and passed) courses
3. Aggregate credits by student and category
4. Compare against program requirements
5. Compute progress and graduation eligibility
6. Check what categories are the students missing
7. Check which classes are the easiest and hardest
"""

import pandas as pd
from google.colab import files

uploaded = files.upload()
students = pd.read_csv("dhsstudents.csv")
courses = pd.read_csv("dhscourses.csv")
program_requirements = pd.read_csv("dhsprogram_requirements.csv")
enrollments = pd.read_csv("dhsenrollments.csv")

Saving dhsenrollments.csv to dhsenrollments (12).csv
Saving dhsprogram_requirements.csv to dhsprogram_requirements (12).csv
Saving dhscourses.csv to dhscourses (12).csv
Saving dhsstudents.csv to dhsstudents (12).csv


In [ ]:
#checking to see if students dataframe is clean and what I expected it to be.
#realized that the name is just the student ID number so I dropped that column.

students.head()
students = students.drop(columns=["Name"])
students.head()

,Student_ID,Program
0,1001,STB
1,1002,MA Theology
2,1003,Master of Divinity
3,1004,Master of Divinity
4,1005,MA Thomistic Studies


In [ ]:
#make sure there are no null values
students.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Student_ID  70 non-null     int64 
 1   Program     70 non-null     object
dtypes: int64(1), object(1)
memory usage: 1.2+ KB


In [ ]:
#same thing that I did with students dataframe, making sure everything is what I expected it to be
#I will be repeating this for all dataframes
courses.head()

,Course_ID,Course_Name,Category,Credits
0,SYS101,Nature and Method of Theology,Systematic Theology,3
1,SYS102,Triune God,Systematic Theology,3
2,SYS103,Creation and the Human Person,Systematic Theology,3
3,SYS104,Christology,Systematic Theology,3
4,MOR101,Moral Life I,Moral Theology,3


In [ ]:
# again, checking if there are null values
courses.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Course_ID    16 non-null     object
 1   Course_Name  16 non-null     object
 2   Category     16 non-null     object
 3   Credits      16 non-null     int64 
dtypes: int64(1), object(3)
memory usage: 644.0+ bytes


In [ ]:
program_requirements.head()

,Program,Category,Required_Credits
0,MA Theology,Systematic Theology,12
1,MA Theology,Moral Theology,6
2,MA Theology,Scripture,6
3,MA Theology,Church History,6
4,MA Theology,Electives,6


In [ ]:
program_requirements.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Program           27 non-null     object
 1   Category          27 non-null     object
 2   Required_Credits  27 non-null     int64 
dtypes: int64(1), object(2)
memory usage: 780.0+ bytes


In [ ]:
enrollments.head()

,Student_ID,Course_ID,Grade,Completed
0,1001,ELEC1,NaN,0
1,1001,LIT101,NaN,0
2,1001,ELEC2,2.10,1
3,1001,SCR102,3.09,1
4,1001,HIS102,NaN,0


In [ ]:
enrollments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 526 entries, 0 to 525
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Student_ID  526 non-null    int64  
 1   Course_ID   526 non-null    object 
 2   Grade       259 non-null    float64
 3   Completed   526 non-null    int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 16.6+ KB


In [ ]:
#merging student and enrollments dataframes to see what student has taken a course and
#what grade they got on the course.
#This gives us a better idea on what classes are the students taking and what grade they have
#conveys us a narrative of what a typical student in the DHS have taken.
student_enrollments = enrollments.merge(students,on="Student_ID")
student_enrollments.head()


,Student_ID,Course_ID,Grade,Completed,Program
0,1001,ELEC1,NaN,0,STB
1,1001,LIT101,NaN,0,STB
2,1001,ELEC2,2.10,1,STB
3,1001,SCR102,3.09,1,STB
4,1001,HIS102,NaN,0,STB


In [ ]:
#combined student_enrollments with courses on "Course_ID".
#This will get me to understand what each student have taken and course info
#and it will help me understand if they are on the right track to graduating.
combined_df = student_enrollments.merge(courses,on="Course_ID")
combined_df.head()

,Student_ID,Course_ID,Grade,Completed,Program,Course_Name,Category,Credits
0,1001,ELEC1,NaN,0,STB,Elective 1,Electives,3
1,1001,LIT101,NaN,0,STB,Liturgiology,Liturgical/Sacramental Theology,3
2,1001,ELEC2,2.10,1,STB,Elective 2,Electives,3
3,1001,SCR102,3.09,1,STB,Synoptic Gospels,Scripture,3
4,1001,HIS102,NaN,0,STB,Church History II,Church History,3


**Who is eligible to graduate?**

1.   Must exceed minimum of credits depending on Degree
2.   Past required classes depending on Degree



In [ ]:
#Shows students who have taken a course that they completed and passed the course with a 2.0 (C-grade).
#I would assume that a student can receive credit if they have a least a C grade and fully completed the course.
#This helps us go deeper into understanding the narrative of students as we know more about their performance in
#certain classes.
valid_courses = combined_df[(combined_df["Completed"] == 1) & (combined_df["Grade"] >= 2.0)]
valid_courses.head()

,Student_ID,Course_ID,Grade,Completed,Program,Course_Name,Category,Credits
2,1001,ELEC2,2.10,1,STB,Elective 2,Electives,3
3,1001,SCR102,3.09,1,STB,Synoptic Gospels,Scripture,3
6,1001,SYS104,3.20,1,STB,Christology,Systematic Theology,3
8,1002,HIS102,3.88,1,MA Theology,Church History II,Church History,3
12,1002,SCR102,3.56,1,MA Theology,Synoptic Gospels,Scripture,3


In [ ]:
#This shows students that have passed a course, how many credits, and what category they accomplishing towards
#aggregated with the sum to show total credits.
student_credits = valid_courses.groupby(["Student_ID", "Program", "Category"])["Credits"].sum().reset_index()
student_credits.head()

,Student_ID,Program,Category,Credits
0,1001,STB,Electives,3
1,1001,STB,Scripture,3
2,1001,STB,Systematic Theology,3
3,1002,MA Theology,Church History,3
4,1002,MA Theology,Scripture,3


In [ ]:
#I merged students with program_requirments to get the full narrative on what a student
#is required to graduate. Again, this gives us a narrative and clearer picture on how many
#required credits are needed in a certain category to accomplish this part of the coursework.
student_requirements = students.merge(program_requirements,on="Program",how="left")
student_requirements.head()

,Student_ID,Program,Category,Required_Credits
0,1001,STB,Systematic Theology,15
1,1001,STB,Moral Theology,9
2,1001,STB,Scripture,18
3,1001,STB,Church History,6
4,1001,STB,Electives,15


In [ ]:
#The progress dataframe is a merging student_credits with student_requirements.
#This is done with the "Student_ID, "Program", and "Category" attributes and then we filled 0s with whatever credits had "NaN"
#The purpose is to see how many credits a student have accomplished
#and compare that to how many credits they need to fulfill.
progress = student_requirements.merge(student_credits,on=["Student_ID", "Program", "Category"],how="left")
progress["Credits"] = progress["Credits"].fillna(0)
progress.head()

,Student_ID,Program,Category,Required_Credits,Credits
0,1001,STB,Systematic Theology,15,3.0
1,1001,STB,Moral Theology,9,0.0
2,1001,STB,Scripture,18,3.0
3,1001,STB,Church History,6,0.0
4,1001,STB,Electives,15,3.0


In [ ]:
#I made a true/false to see if a student had met the requirements for the certain category of coursework.
#It is easier to understand if the coursework category has been met or not through a true or false.
progress["Meets_Requirement"] = (progress["Credits"] >= progress["Required_Credits"])
progress

,Student_ID,Program,Category,Required_Credits,Credits,Meets_Requirement
0,1001,STB,Systematic Theology,15,3.0,False
1,1001,STB,Moral Theology,9,0.0,False
2,1001,STB,Scripture,18,3.0,False
3,1001,STB,Church History,6,0.0,False
4,1001,STB,Electives,15,3.0,False
...,...,...,...,...,...,...
373,1070,MA Theology,Systematic Theology,12,0.0,False
374,1070,MA Theology,Moral Theology,6,0.0,False
375,1070,MA Theology,Scripture,6,0.0,False
376,1070,MA Theology,Church History,6,0.0,False


In [ ]:
#I grouped the progress by the student's ID and dropped all columns other than "Meets_Requirement"
#This shows if a student has met the requirement to graduate
graduation_status = progress.groupby("Student_ID")["Meets_Requirement"].all().reset_index()
graduation_status.head()

,Student_ID,Meets_Requirement
0,1001,False
1,1002,False
2,1003,False
3,1004,False
4,1005,False


In [ ]:
#I went a bit further by going back to progress and found the progress of fulfilling thier courseworks per category.
#I did this by diving someone's credits with their required credits on a certain category of course work.
#Clipped by one to take away those who have over a 1.
progress["Category_Progress"] = (progress["Credits"] / progress["Required_Credits"]).clip(upper = 1)
progress.head()


,Student_ID,Program,Category,Required_Credits,Credits,Meets_Requirement,Category_Progress
0,1001,STB,Systematic Theology,15,3.0,False,0.200000
1,1001,STB,Moral Theology,9,0.0,False,0.000000
2,1001,STB,Scripture,18,3.0,False,0.166667
3,1001,STB,Church History,6,0.0,False,0.000000
4,1001,STB,Electives,15,3.0,False,0.200000


In [ ]:
#sanity check to see if there are rows with "Category_Progress" is greater than 1
progress[progress["Category_Progress"] > 1]


,Student_ID,Program,Category,Required_Credits,Credits,Meets_Requirement,Category_Progress


In [ ]:
#I am shifting the dataframe from a category level to a program level.
#This is done through grouping the progress df with "Student_ID" and "Program".
#I found the mean of the category progress and renamed it to "Graduation_Progress".
#This dataframe is now seen on the level of a program as we are able to look now
#on the progress of a student hollistically.
student_progress = progress.groupby(["Student_ID", "Program"])["Category_Progress"].mean().reset_index().rename(columns={"Category_Progress":"Graduation_Progress"})
student_progress.head()

,Student_ID,Program,Graduation_Progress
0,1001,STB,0.113333
1,1002,MA Theology,0.200000
2,1003,Master of Divinity,0.140212
3,1004,Master of Divinity,0.055556
4,1005,MA Thomistic Studies,0.000000


In [ ]:
#I merged student progress and graduation status with the "Student_ID".
#Now, we can see a fuller picture if the person has met the requirement or not.
student_progress = student_progress.merge(graduation_status, on="Student_ID")
student_progress.head()

,Student_ID,Program,Graduation_Progress,Meets_Requirement
0,1001,STB,0.113333,False
1,1002,MA Theology,0.200000,False
2,1003,Master of Divinity,0.140212,False
3,1004,Master of Divinity,0.055556,False
4,1005,MA Thomistic Studies,0.000000,False


The dataframe student_progress shows us the progress of a student to graduate their selected program!

**Students and their Missing Category**

In [ ]:
#This dataframe shows students with zero credits in a required category
missing_categories = progress[progress["Credits"] == 0]
missing_categories = missing_categories.groupby(["Student_ID", "Program"])["Category"].apply(list).reset_index().rename(columns={"Category": "Missing_Categories"})
missing_categories.head()

,Student_ID,Program,Missing_Categories
0,1001,STB,"[Moral Theology, Church History]"
1,1002,MA Theology,"[Systematic Theology, Moral Theology, Electives]"
2,1003,Master of Divinity,"[Liturgical/Sacramental Theology, Moral Theolo..."
3,1004,Master of Divinity,"[Systematic Theology, Liturgical/Sacramental T..."
4,1005,MA Thomistic Studies,"[Systematic Theology, Moral Theology, Thesis D..."


*Conclusion*

Seeing the missing categories can give us an insight on a student. If they are missing some of these categories, we will try to give them a schedule for next semester and so forth that will help tackle the categories that are needed to be fulfilled in order to graduate. I would see the missing categories and I would start looking for availability in these courses and get them signed up for that course.

**What courses are most difficult or easy?**

Seeing this statistic can help us see what needs to be changed in regards to the structure and content of the courses based on the performance and completion of the course from past history.

In [ ]:
#calculating the percentage of students who completed courses within each program
combined_df.groupby("Program")["Completed"].mean()

,Completed
Program,
MA Theology,0.480000
MA Thomistic Studies,0.503937
Master of Divinity,0.458716
STB,0.522124
STL,0.493506


In [ ]:
#which courses have the lowest vs highest average student performance?
combined_df.groupby("Course_Name")["Grade"].mean().sort_values()

,Grade
Course_Name,
Church History II,2.100000
Church History I,2.133500
Elective 1,2.164545
Nature and Method of Theology,2.368824
Pastoral Ministry,2.397857
Christology,2.408421
Synoptic Gospels,2.604118
Pentateuch,2.620000
Intro to Canon Law,2.622105


In [ ]:
#which courses do students drop or fail to finish most often?
combined_df.groupby("Course_Name")["Completed"].mean().sort_values()

,Completed
Course_Name,
Moral Life II,0.363636
Elective 1,0.379310
Pentateuch,0.405405
Triune God,0.405405
Moral Life I,0.432432
Pastoral Ministry,0.451613
Church History II,0.485714
Church History I,0.487805
Liturgiology,0.500000


In [ ]:
#brings fuller picture of what courses have a high grade average
#and on average how many students completed the course
combined_df.groupby("Course_Name").agg({"Grade": "mean","Completed": "mean"}).sort_values("Grade")

,Grade,Completed
Course_Name,,
Church History II,2.100000,0.485714
Church History I,2.133500,0.487805
Elective 1,2.164545,0.379310
Nature and Method of Theology,2.368824,0.548387
Pastoral Ministry,2.397857,0.451613
Christology,2.408421,0.655172
Synoptic Gospels,2.604118,0.531250
Pentateuch,2.620000,0.405405
Intro to Canon Law,2.622105,0.575758


*Conclusion*

I would then report to the dean about my findings and suggest to meet with the professors with low grade and completion to help them restructure the class. There are so many variables to why a course might not have high completion and grades. This could be because the content is not engaging, assignments are difficult, or even the professor may not be the best at teaching this content.

High grade and high completion can also be seen as good. The class might be seen as something that is clear cut and palpable for students to learn. There might also be a lot of support from professors and TAs. I can also see the concern that maybe the content is not as rigorous as we'd like.

High grade and low completion means that people are dropping out of the course. I would suggest again evaluating how rigorous the course can be. I would assume people would see this course as too hard and is not possible given their schedule that could potentially have them lose hope on taking the class.

High completion and low grade might mean that the course is poorly structured. There could be possibility that the course content is quite difficult but the content has convicted students enough that they would persist. I would keep my eye on this course, but I do believe that post-grad courses should have a sense of difficulty into them.
